# 10. Bloom 与 Counting Filter：怎样观察误报，并在删除时避免制造假阴性？

## 面试回答主线

Bloom Filter 用 k 个哈希把 key 映射到 bit array，查询时只要有一位为零就一定不存在，全部为一则是“可能存在”。它节省内存但允许 false positive，不允许 false negative；误报率由 bit 数、哈希数和插入量决定。普通 Bloom 不能安全删除，因为清除一个共享 bit 会让其他 key 变成假阴性。Counting Bloom 将 bit 换成 counter，插入递增、删除递减，只有 counter 全部大于零才判定可能存在。面试时我会用真实风控 key 输出哈希位置、bit 碰撞、误报与删除轨迹。生产还要处理 counter 饱和、并发、删除不存在 key 和持久化重建。

## 1. 真实案例：八个需要快速拦截的风控键

成员集合覆盖欺诈用户、滥用 IP、泄露密钥和冻结订单；查询集同时包含已知成员与六个正常键。精确 set 是正确性基线，Bloom 使用 24 bit 和 3 个稳定哈希以真实产生少量误报。

In [1]:
from pprint import pprint  # 导入结构化打印工具展示过滤器输入、哈希和结果
import hashlib  # 导入稳定哈希函数实现可复现 bit 位置
members = ["fraud:U17", "fraud:U23", "abuse:IP-8", "abuse:IP-9", "key:AKIA-1", "key:AKIA-2", "order:O77", "order:O88"]  # 定义八个真实风控拦截键
queries = ["fraud:U17", "key:AKIA-2", "order:O88", "fraud:U99", "abuse:IP-7", "key:AKIA-9", "order:O66", "user:U17", "refund:O77"]  # 定义三个成员查询和六个非成员查询
bit_count = 24  # 设置较小 bit array 以展示实际哈希碰撞和误报
hash_count = 3  # 设置三个独立哈希位置平衡误报与计算
preview = [{"key": key, "类别": key.split(":", 1)[0], "应拦截": key in set(members)} for key in queries]  # 汇总在线查询的真实业务语义
print("Bloom Filter 查询预览：")  # 输出真实案例标题
pprint(preview, sort_dicts=False)  # 展示成员与正常请求的期望判断

Bloom Filter 查询预览：
[{'key': 'fraud:U17', '类别': 'fraud', '应拦截': True},
 {'key': 'key:AKIA-2', '类别': 'key', '应拦截': True},
 {'key': 'order:O88', '类别': 'order', '应拦截': True},
 {'key': 'fraud:U99', '类别': 'fraud', '应拦截': False},
 {'key': 'abuse:IP-7', '类别': 'abuse', '应拦截': False},
 {'key': 'key:AKIA-9', '类别': 'key', '应拦截': False},
 {'key': 'order:O66', '类别': 'order', '应拦截': False},
 {'key': 'user:U17', '类别': 'user', '应拦截': False},
 {'key': 'refund:O77', '类别': 'refund', '应拦截': False}]


## 2. Baseline（基线）：精确 set 无误报，但内存随 key 数增长

精确集合能安全插入、查询和删除，每个完整字符串都要保留。它为 Bloom 的误报/假阴性统计提供 ground truth。

In [2]:
exact_set = set(members)  # 用完整字符串建立无误差成员集合
baseline_rows = []  # 收集九个查询的精确判断
for key in queries:  # 遍历成员和非成员混合查询
    present = key in exact_set  # 执行哈希集合的精确 membership 查询
    baseline_rows.append({"key": key, "精确存在": present, "决策": "拦截" if present else "放行"})  # 保存逐 key 正确基线
print("Exact set 查询结果：")  # 标注当前输出属于精确基线
pprint(baseline_rows, sort_dicts=False)  # 展示精确集合没有误报与漏报
print({"保存完整key数量": len(exact_set), "误报": 0, "假阴性": 0})  # 汇总基线正确性与 key 级内存规模

Exact set 查询结果：
[{'key': 'fraud:U17', '精确存在': True, '决策': '拦截'},
 {'key': 'key:AKIA-2', '精确存在': True, '决策': '拦截'},
 {'key': 'order:O88', '精确存在': True, '决策': '拦截'},
 {'key': 'fraud:U99', '精确存在': False, '决策': '放行'},
 {'key': 'abuse:IP-7', '精确存在': False, '决策': '放行'},
 {'key': 'key:AKIA-9', '精确存在': False, '决策': '放行'},
 {'key': 'order:O66', '精确存在': False, '决策': '放行'},
 {'key': 'user:U17', '精确存在': False, '决策': '放行'},
 {'key': 'refund:O77', '精确存在': False, '决策': '放行'}]
{'保存完整key数量': 8, '误报': 0, '假阴性': 0}


## 3. 手写 Bloom Filter：稳定多哈希、bit 写入与碰撞

每个 key 的三个位置由 `seed|key` 的 BLAKE2b 摘要决定。插入只把 bit 置一；查询读取三位并做 all。下面输出八个成员的哈希位置和最终 bit array。

In [3]:
def hash_positions(key, size=bit_count, count=hash_count):  # 为一个风控键计算多个稳定 bit 位置
    positions = []  # 初始化当前 key 的哈希位置列表
    for seed in range(count):  # 使用行号构造三个独立哈希函数
        digest = hashlib.blake2b(f"{seed}|{key}".encode("utf-8"), digest_size=8).digest()  # 生成可复现八字节摘要
        positions.append(int.from_bytes(digest, "big") % size)  # 将摘要映射到固定 bit array 范围
    return positions  # 返回插入与查询共同使用的位置
class BloomFilter:  # 定义不支持安全删除的普通 Bloom Filter
    def __init__(self, size, hashes):  # 初始化 bit 数和哈希数量
        self.size = size  # 保存 bit array 长度
        self.hashes = hashes  # 保存每个 key 的哈希位置数量
        self.bits = [0] * size  # 分配初始全零 bit array
    def add(self, key):  # 把一个 key 插入 Bloom Filter
        positions = hash_positions(key, self.size, self.hashes)  # 计算当前 key 的全部 bit 位置
        for position in positions:  # 遍历三个哈希桶
            self.bits[position] = 1  # 将对应 bit 幂等设置为一
        return positions  # 返回位置供碰撞审计
    def contains(self, key):  # 查询一个 key 是否可能存在
        positions = hash_positions(key, self.size, self.hashes)  # 重算与插入完全相同的 bit 位置
        values = [self.bits[position] for position in positions]  # 读取三个 bit 当前状态
        return all(values), positions, values  # 全为一表示可能存在否则一定不存在
bloom = BloomFilter(bit_count, hash_count)  # 创建二十四 bit、三哈希的普通过滤器
member_positions = {key: bloom.add(key) for key in members}  # 插入八个风控成员并保存哈希位置
print("成员 key 的三个哈希位置：")  # 输出核心机制中间量标题
pprint(member_positions, sort_dicts=False)  # 展示共享 bit 与碰撞来源
print("最终 bit array：", bloom.bits)  # 展示完整固定内存状态

成员 key 的三个哈希位置：
{'fraud:U17': [0, 21, 13],
 'fraud:U23': [20, 7, 22],
 'abuse:IP-8': [8, 20, 11],
 'abuse:IP-9': [2, 12, 17],
 'key:AKIA-1': [13, 20, 5],
 'key:AKIA-2': [19, 1, 19],
 'order:O77': [21, 15, 17],
 'order:O88': [6, 20, 18]}
最终 bit array： [1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0]


## 4. 逐查询误报与假阴性统计

Bloom 的 positive 只能表示“可能存在”。下面与 exact set 逐 key 比较，保留查询位置和 bit 值；在当前参数下 `fraud:U99` 会发生真实误报，但所有已插入成员都不会漏掉。

In [4]:
bloom_rows = []  # 收集九个查询的 Bloom 判断与 ground truth
for key in queries:  # 遍历相同在线查询集合
    predicted, positions, values = bloom.contains(key)  # 读取三个 bit 并生成可能存在判断
    actual = key in exact_set  # 从精确集合取得成员真值
    bloom_rows.append({"key": key, "位置": positions, "bit值": values, "Bloom可能存在": predicted, "真实存在": actual, "误报": predicted and not actual, "假阴性": actual and not predicted})  # 保存逐 key 错误类型
false_positives = [row["key"] for row in bloom_rows if row["误报"]]  # 收集非成员却被全部 bit 命中的 key
false_negatives = [row["key"] for row in bloom_rows if row["假阴性"]]  # 收集被错误放行的已插入成员
print("Bloom Filter 逐查询结果：")  # 输出核心方案结果标题
pprint(bloom_rows, sort_dicts=False)  # 展示误报由哪些共享 bit 组成
print({"误报key": false_positives, "假阴性key": false_negatives, "非成员查询数": sum(key not in exact_set for key in queries)})  # 汇总受控参数下的错误行为

Bloom Filter 逐查询结果：
[{'key': 'fraud:U17',
  '位置': [0, 21, 13],
  'bit值': [1, 1, 1],
  'Bloom可能存在': True,
  '真实存在': True,
  '误报': False,
  '假阴性': False},
 {'key': 'key:AKIA-2',
  '位置': [19, 1, 19],
  'bit值': [1, 1, 1],
  'Bloom可能存在': True,
  '真实存在': True,
  '误报': False,
  '假阴性': False},
 {'key': 'order:O88',
  '位置': [6, 20, 18],
  'bit值': [1, 1, 1],
  'Bloom可能存在': True,
  '真实存在': True,
  '误报': False,
  '假阴性': False},
 {'key': 'fraud:U99',
  '位置': [22, 15, 6],
  'bit值': [1, 1, 1],
  'Bloom可能存在': True,
  '真实存在': False,
  '误报': True,
  '假阴性': False},
 {'key': 'abuse:IP-7',
  '位置': [0, 11, 16],
  'bit值': [1, 1, 0],
  'Bloom可能存在': False,
  '真实存在': False,
  '误报': False,
  '假阴性': False},
 {'key': 'key:AKIA-9',
  '位置': [23, 6, 18],
  'bit值': [0, 1, 1],
  'Bloom可能存在': False,
  '真实存在': False,
  '误报': False,
  '假阴性': False},
 {'key': 'order:O66',
  '位置': [4, 0, 10],
  'bit值': [0, 1, 0],
  'Bloom可能存在': False,
  '真实存在': False,
  '误报': False,
  '假阴性': False},
 {'key': 'user:U17',
  '位置': [10, 19, 7],

## 5. 结果解读：Counting Bloom 用 counter 支持引用计数式删除

Counting Filter 用整数 counter 替代 bit。这里把 `fraud:U17` 插入两次：删除一次后仍应存在，删除第二次才消失；其他共享桶成员保持可查询。

In [5]:
class CountingBloomFilter:  # 定义支持引用计数删除的 Counting Bloom Filter
    def __init__(self, size, hashes):  # 初始化 counter array 和哈希数量
        self.size = size  # 保存 counter 数量
        self.hashes = hashes  # 保存每个 key 的独立哈希数量
        self.counters = [0] * size  # 分配初始全零整数 counters
    def add(self, key):  # 插入一个 key 并增加全部引用计数
        for position in hash_positions(key, self.size, self.hashes):  # 遍历当前 key 的三个稳定位置
            self.counters[position] += 1  # 增加共享桶计数而不是只置一
    def remove(self, key):  # 删除一个已知存在 key 的一次引用
        positions = hash_positions(key, self.size, self.hashes)  # 计算需要递减的三个 counter
        if not all(self.counters[position] > 0 for position in positions):  # 防止删除明显不存在 key 造成 counter 下溢
            return False  # 拒绝无法证明安全的删除操作
        for position in positions:  # 遍历三个共享 counters
            self.counters[position] -= 1  # 只减少当前 key 的一次引用计数
        return True  # 返回删除已应用的回执
    def contains(self, key):  # 查询一个 key 是否可能仍有引用
        positions = hash_positions(key, self.size, self.hashes)  # 计算当前 key 的三个 counter 位置
        return all(self.counters[position] > 0 for position in positions)  # 全部 counter 为正才判定可能存在
counting = CountingBloomFilter(bit_count, hash_count)  # 创建与普通 Bloom 相同宽度和哈希数的 counting 版本
for key in members:  # 插入完整风控成员集合
    counting.add(key)  # 为每个成员增加三个 counter
counting.add("fraud:U17")  # 模拟同一风控规则被两个来源引用
before_delete = counting.contains("fraud:U17")  # 检查双引用成员删除前存在状态
first_delete_applied = counting.remove("fraud:U17")  # 删除第一个来源的引用
after_one_delete = counting.contains("fraud:U17")  # 确认剩余一个引用仍保持可能存在
second_delete_applied = counting.remove("fraud:U17")  # 删除第二个来源的引用
after_two_deletes = counting.contains("fraud:U17")  # 确认全部引用移除后不再命中
other_members_safe = {key: counting.contains(key) for key in members if key != "fraud:U17"}  # 检查共享 counters 的其他成员没有被误删
print({"删除前": before_delete, "删除一次后": after_one_delete, "删除两次后": after_two_deletes, "其他成员仍存在": other_members_safe})  # 展示引用计数删除轨迹

{'删除前': True, '删除一次后': True, '删除两次后': False, '其他成员仍存在': {'fraud:U23': True, 'abuse:IP-8': True, 'abuse:IP-9': True, 'key:AKIA-1': True, 'key:AKIA-2': True, 'order:O77': True, 'order:O88': True}}


## 6. 失败案例与修正：普通 Bloom 清 bit 会让共享成员假阴性

若直接把 `fraud:U17` 的三个 bit 清零，`key:AKIA-1` 与 `order:O77` 共用的 bit 也会消失，两个仍在集合中的成员变成假阴性。Counting Bloom 只递减 counter，其他 key 的引用仍能保持 counter 大于零。

In [6]:
broken_bits = bloom.bits.copy()  # 复制已包含全部成员的普通 Bloom bit array
for position in hash_positions("fraud:U17"):  # 遍历被删除 key 的三个 bit
    broken_bits[position] = 0  # 错误地直接清除可能由多个成员共享的 bit
def contains_in_bits(key, bits):  # 在指定 bit 快照上执行普通 Bloom 查询
    return all(bits[position] == 1 for position in hash_positions(key))  # 检查 key 的三个位置是否仍为一
lost_members = [key for key in members if key != "fraud:U17" and not contains_in_bits(key, broken_bits)]  # 找出因共享 bit 被清除而产生的无辜假阴性
counting_preserved = [key for key, present in other_members_safe.items() if present]  # 收集 counting 删除后仍安全存在的其他成员
print({"失败_普通Bloom删除后丢失": lost_members, "被删除key位置": hash_positions("fraud:U17"), "修正_Counting保留成员数": len(counting_preserved), "修正后保留": counting_preserved})  # 展示 bit 删除事故与 counter 修复

{'失败_普通Bloom删除后丢失': ['key:AKIA-1', 'order:O77'], '被删除key位置': [0, 21, 13], '修正_Counting保留成员数': 7, '修正后保留': ['fraud:U23', 'abuse:IP-8', 'abuse:IP-9', 'key:AKIA-1', 'key:AKIA-2', 'order:O77', 'order:O88']}


## 7. 生产差距与最小回归检查

生产设计应根据目标误报率计算 m 与 k，监控实际插入量偏离容量规划；哈希需稳定且防止恶意碰撞。Counting counter 可能溢出，删除不存在 key 也不能仅凭 filter 自证安全，通常还需权威引用账本或周期重建。下面的断言只验证本实验的成员数、误报、无原生假阴性、引用删除和共享 bit 事故。

In [7]:
assert len(members) >= 6 and len(queries) >= 6  # 确认真实成员与查询数量满足逐样本教学要求
assert len(false_positives) > 0  # 确认受控 bit 容量真实产生可观察 Bloom 误报
assert false_negatives == []  # 确认只插入不删除时普通 Bloom 不会漏掉成员
assert before_delete and after_one_delete and not after_two_deletes  # 确认 counting filter 正确处理双引用的两次删除
assert all(other_members_safe.values())  # 确认 counting 删除没有破坏任何其他成员
assert len(lost_members) > 0  # 确认普通 Bloom 直接清 bit 真实制造共享成员假阴性
print("回归检查通过：多哈希 bit、真实误报、Counting 删除与共享 bit 假阴性均已验证。")  # 输出最终验收结论

回归检查通过：多哈希 bit、真实误报、Counting 删除与共享 bit 假阴性均已验证。
